# Sensitivity of the Accept/Reject/Uncertain breakdown to support thresholds

How much does the categorisation in `04_rule_categorization.ipynb` depend on
the mining thresholds (`min_undesired_support`, `min_desired_support`)?

For each dataset we re-mine under three configurations:

- `base` — the calibrated thresholds used in the main results.
- `tight` — higher per-class support thresholds (~+25 %), fewer rules.
- `loose` — lower per-class support thresholds (~-25 %), more rules.

Then we recompute bootstrap percentile 95 % CIs (B=500, seed=42) and
categorise each rule at `tau = 0` separately for the uplift CI and the
realistic rule gain CI.

**Output**: `notebooks/inference_studies/results/threshold_sensitivity.csv`.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

# Import the shared dataset loader (canonical preprocessing for the four
# benchmark datasets used throughout this folder).
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))
from _datasets import load_all, load_telco, load_bank_marketing, load_employee_attrition, load_credit_card_default  # noqa: E402

# Outputs for this notebook (kept under the notebook folder so the user can
# inspect them locally without touching the article's canonical CSVs).
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'inference_studies' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root:   {REPO_ROOT}')
print(f'Results dir: {RESULTS_DIR.relative_to(REPO_ROOT)}')


In [ ]:
from collections import Counter
from dataclasses import replace
from action_rules import ActionRules
from _datasets import ALL_LOADERS, DatasetConfig

OUT_CSV = RESULTS_DIR / 'threshold_sensitivity.csv'

N_BOOTSTRAP = 500
SEED = 42
THRESHOLD = 0.0


In [ ]:
# Per-dataset tight/loose support perturbations.
PERTURBATIONS = {
    'Telco Customer Churn': {
        'tight': {'min_undesired_support': 270, 'min_desired_support': 135},
        'loose': {'min_undesired_support': 170, 'min_desired_support': 85},
    },
    'UCI Bank Marketing': {
        'tight': {'min_undesired_support': 220, 'min_desired_support': 220},
        'loose': {'min_undesired_support': 150, 'min_desired_support': 150},
    },
    'IBM Employee Attrition': {
        'tight': {'min_undesired_support': 22, 'min_desired_support': 22},
        'loose': {'min_undesired_support': 14, 'min_desired_support': 14},
    },
    'Taiwan Credit Card Default': {
        'tight': {'min_undesired_support': 575, 'min_desired_support': 575},
        'loose': {'min_undesired_support': 350, 'min_desired_support': 350},
    },
}


In [ ]:
def mine_and_categorise(cfg: DatasetConfig) -> dict:
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    ar.fit(
        cfg.df,
        stable_attributes=cfg.stable_attributes,
        flexible_attributes=cfg.flexible_attributes,
        target=cfg.target,
        target_undesired_state=cfg.target_undesired_state,
        target_desired_state=cfg.target_desired_state,
        use_sparse_matrix=cfg.use_sparse_matrix,
    )
    n_rules = len(ar.output.action_rules) if ar.output is not None else 0
    if n_rules == 0:
        return {
            'n_rules': 0,
            'uplift_accept': 0, 'uplift_uncertain': 0, 'uplift_reject': 0,
            'gain_accept': 0, 'gain_uncertain': 0, 'gain_reject': 0,
        }
    counts = {}
    for metric in ('uplift', 'realistic_rule_gain'):
        results = ar.confidence_intervals(
            cfg.df,
            method='bootstrap',
            confidence_level=0.95,
            threshold=THRESHOLD,
            metric=metric,
            n_bootstrap=N_BOOTSTRAP,
            random_state=SEED,
        )
        counts[metric] = Counter(r.category.value for r in results if r.category is not None)
    return {
        'n_rules': n_rules,
        'uplift_accept':   counts['uplift'].get('accept', 0),
        'uplift_uncertain': counts['uplift'].get('uncertain', 0),
        'uplift_reject':   counts['uplift'].get('reject', 0),
        'gain_accept':     counts['realistic_rule_gain'].get('accept', 0),
        'gain_uncertain':  counts['realistic_rule_gain'].get('uncertain', 0),
        'gain_reject':     counts['realistic_rule_gain'].get('reject', 0),
    }


In [ ]:
rows = []
for ds_name, loader in ALL_LOADERS.items():
    print(f'=== {ds_name} ===')
    base_cfg = loader()
    configs = {'base': base_cfg}
    for label, overrides in PERTURBATIONS[ds_name].items():
        configs[label] = replace(base_cfg, **overrides)

    for cfg_label, cfg in configs.items():
        print(
            f'  [{cfg_label}] support={cfg.min_undesired_support}/'
            f'{cfg.min_desired_support} conf={cfg.min_undesired_confidence}'
        )
        res = mine_and_categorise(cfg)
        res.update({
            'dataset': ds_name,
            'config': cfg_label,
            'min_undesired_support': cfg.min_undesired_support,
            'min_desired_support': cfg.min_desired_support,
            'min_undesired_confidence': cfg.min_undesired_confidence,
            'min_desired_confidence': cfg.min_desired_confidence,
        })
        rows.append(res)
        print(
            f'    -> n={res["n_rules"]}, uplift A/U/R='
            f'{res["uplift_accept"]}/{res["uplift_uncertain"]}/{res["uplift_reject"]},'
            f' gain A/U/R={res["gain_accept"]}/{res["gain_uncertain"]}/{res["gain_reject"]}'
        )

df = pd.DataFrame(rows)[[
    'dataset', 'config',
    'min_undesired_support', 'min_desired_support',
    'min_undesired_confidence', 'min_desired_confidence',
    'n_rules',
    'uplift_accept', 'uplift_uncertain', 'uplift_reject',
    'gain_accept', 'gain_uncertain', 'gain_reject',
]]
df.to_csv(OUT_CSV, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)}')
df


## Interpretation

If the Accept / Reject split is roughly the same across `tight`, `base` and
`loose`, the categorisation is **robust to mining hyperparameters** — the
rules that pass at base thresholds pass at tighter thresholds, and most
additional rules that appear under looser thresholds also fall into the
Accept bucket.

A large shift in the percentages — particularly a jump in Reject rules under
`loose` — signals that the looser support cutoff is admitting underpowered
rules whose CIs straddle zero.
